# 🛡️ WIA1006 — Catfish Detector | Group 7
## V15.0 — SCANNER DEFINITIVELY FIXED

> **Root cause of identical scanner results (all versions V10–V14):**
> `SelectFromModel` dropped **ALL 12 slider-controlled features** (0 survived).
> After `selector.transform()`, every input vector was identical — so every prediction was identical.
> **V15 Fix:** `SelectFromModel` removed entirely. Models train on the full post-scaling feature space.
> Scanner uses `scaler.transform()` only — slider values reach the models unchanged.
> Slider ranges corrected to match real dataset bounds (`message_sent_count` max=100, `app_usage_time_min` max=300).
> Default slider values updated to valid in-range examples.

## 🔗 Cell 1 — Mount Google Drive
> Run FIRST every session.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 | Mount Google Drive — run FIRST every session
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

CSV_PATH = ('/content/drive/MyDrive/Colab Notebooks/'
            'WIA1006_GRP_7_PROJECT/dating_app_behavior_dataset.csv')

if os.path.exists(CSV_PATH):
    print(f'✅ Drive mounted. Dataset: {os.path.getsize(CSV_PATH)/1e6:.1f} MB')
    print(f'   {CSV_PATH}')
else:
    print('❌ File not found. Searching...')
    import subprocess
    r = subprocess.run(['find','/content/drive/MyDrive',
                        '-name','dating_app_behavior_dataset.csv'],
                       capture_output=True, text=True)
    found = r.stdout.strip()
    if found:
        print(f'   Found at: {found}')
        print('   → Update CSV_PATH above and re-run.')
    else:
        print('   Not found — please re-upload to Drive.')


## 📦 Cell 2 — Install Libraries

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 | Install Libraries
# ═══════════════════════════════════════════════════════════════
!pip install -q imbalanced-learn xgboost scikit-learn \
               pandas numpy matplotlib seaborn joblib scipy
print('✅ All libraries installed.')


## 📚 Cell 3 — Master Imports
> Every downstream cell also imports locally — NameErrors impossible.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 | MASTER IMPORTS — every downstream cell also re-imports
# ═══════════════════════════════════════════════════════════════
import os, gc, warnings
import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import matplotlib; import seaborn as sns
from scipy.stats               import zscore
from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import RobustScaler
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.metrics           import (accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay, precision_recall_curve)
from sklearn.linear_model      import LogisticRegression
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import (RandomForestClassifier,
                                        ExtraTreesClassifier)
from sklearn.neural_network    import MLPClassifier
from xgboost                   import XGBClassifier
from imblearn.combine          import SMOTETomek
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':110, 'font.size':9})
print('✅ ALL libraries imported.')
print(f'   pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')


## 📂 Cell 4 — Load & Clean Dataset

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 | Load & Clean Dataset
# ═══════════════════════════════════════════════════════════════
import os, warnings
import pandas as pd, numpy as np
from scipy.stats import zscore
warnings.filterwarnings('ignore')
if 'CSV_PATH' not in dir() or not os.path.exists(CSV_PATH):
    raise RuntimeError('❌ Run Cell 1 first.')

print('📂 Loading...')
df_raw = pd.read_csv(CSV_PATH, engine='python', on_bad_lines='skip',
                      encoding='utf-8', encoding_errors='replace')
print(f'   Raw rows: {len(df_raw):,}')

NUM_RAW = ['message_sent_count','app_usage_time_min',
           'swipe_right_ratio','bio_length','profile_pics_count','age']
for c in NUM_RAW:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

df = df_raw.dropna().reset_index(drop=True)
print(f'   After nulls: {len(df):,}')

zc = [c for c in NUM_RAW if c in df.columns]
df = df[(np.abs(zscore(df[zc].astype(float)))<4).all(axis=1)].reset_index(drop=True)
print(f'   After outliers: {len(df):,}')

n_cat = (df['match_outcome']=='Catfished').sum()
n_gen = len(df)-n_cat
print(f'   Catfished:{n_cat:,} ({n_cat/len(df)*100:.1f}%) | Genuine:{n_gen:,} ({n_gen/len(df)*100:.1f}%)')
print('✅ Dataset ready.')
df.head(3)


## 🔧 Cell 5 — Feature Engineering

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 | Feature Engineering — 12 Behavioural Features
# ═══════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
if 'df' not in dir(): raise RuntimeError('❌ Run Cell 4 first.')

EPS = 1e-6
df['engagement_score'] = df['message_sent_count']/(df['app_usage_time_min']+1)
df['swipe_msg_ratio']  = df['message_sent_count']/(df['swipe_right_ratio']+EPS)
df['msg_per_minute']   = df['message_sent_count']/(df['app_usage_time_min']+EPS)
df['bio_efficiency']   = df['bio_length']/(df['message_sent_count']+1)
df['bio_per_swipe']    = df['bio_length']/(df['swipe_right_ratio']+EPS)
df['bio_per_minute']   = df['bio_length']/(df['app_usage_time_min']+1)
df['swipe_intensity']  = df['swipe_right_ratio']/(df['app_usage_time_min']+EPS)
df['swipe_x_msg']      = df['swipe_right_ratio']*df['message_sent_count']
if 'profile_pics_count' in df.columns:
    df['pic_msg_ratio']   = df['profile_pics_count']/(df['message_sent_count']+1)
    df['pic_swipe_ratio'] = df['profile_pics_count']/(df['swipe_right_ratio']+EPS)
    df['pic_per_minute']  = df['profile_pics_count']/(df['app_usage_time_min']+1)
    print('   + 3 pic features')
df['Target'] = (df['match_outcome']=='Catfished').astype(int)
print(f'✅ Feature engineering done. Columns: {df.shape[1]}')
print(f'   Catfished:{df["Target"].sum():,} | Genuine:{(df["Target"]==0).sum():,}')


## 🧹 Cell 6 — Preprocessing

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 | Preprocessing: Drop → OHE → Correlation → Variance
# ═══════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
from sklearn.feature_selection import VarianceThreshold
if 'df' not in dir() or 'Target' not in df.columns:
    raise RuntimeError('❌ Run Cells 4 & 5 first.')

DROP = ['match_outcome','user_id','Target','location_name',
        'swipe_time_of_day','app_usage_time_label','swipe_right_label']
X_t = df.drop(columns=[c for c in DROP if c in df.columns])
for col in X_t.select_dtypes('object').columns.tolist():
    if X_t[col].nunique()>50: X_t=X_t.drop(columns=[col])

X_ohe = pd.get_dummies(X_t, drop_first=True).astype(float)
print(f'   After OHE: {X_ohe.shape[1]} features')

up = X_ohe.corr().abs()
up = up.where(np.triu(np.ones(up.shape),k=1).astype(bool))
dc = [c for c in up.columns if any(up[c]>0.95)]
if dc: X_ohe=X_ohe.drop(columns=dc); print(f'   Dropped {len(dc)} correlated')

vt    = VarianceThreshold(threshold=0.01)
Xv    = vt.fit_transform(X_ohe)
kept  = X_ohe.columns[vt.get_support()].tolist()
X     = pd.DataFrame(Xv, columns=kept)
y     = df['Target'].reset_index(drop=True)
print(f'✅ Done. Final: {X.shape[0]:,} × {X.shape[1]}')


## ✂️ Cell 7 — Split & Scale

### What is saved here and why:
| Variable | What it is | Used in scanner |
|----------|-----------|----------------|
| `FEATURE_NAMES` | Ordered column names after OHE | Build input DataFrame |
| `NUM_COLS` | Which cols scaler was fitted on | `scaler.transform()` |
| `X_TRAIN_MEDIANS_RAW` | **Median of X_tr BEFORE scaling** | **Scanner baseline** |
| `GENUINE_MEDIANS_RAW` | Median of genuine profiles (unscaled) | 'Load Genuine' button |
| `CATFISH_MEDIANS_RAW` | Median of catfish profiles (unscaled) | 'Load Catfish' button |

> **Why unscaled medians?** Scanner builds a full DataFrame in raw space,
> overrides slider features, then calls `scaler.transform()` exactly once.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 | Train-Test Split + RobustScaler
#
# ROOT CAUSE OF ALL PREVIOUS SCANNER BUGS — NOW FIXED:
#
# V10-V12: base = X_test_arr.mean() [SCALED] → called scaler.transform()
#          again → double-scaled OHE binary cols → same vector every time.
#
# V13: Used scaler.center_/scale_ directly to scale individual features.
#      BUT: SelectFromModel DROPS the features we were updating because
#      their importances are below median. After selector.transform(),
#      the output vector was IDENTICAL regardless of slider inputs.
#
# V14 FIX:
#   Save X_train_df BEFORE scaling as X_TRAIN_MEDIANS_RAW (per-column medians).
#   Scanner starts from these RAW medians, overrides 4 input cols + 8 engineered
#   cols in RAW space, builds a complete DataFrame, calls scaler.transform()
#   ONCE on NUM_COLS, then selector.transform().
#   No index manipulation. No manual scaling math. No double-scaling.
#   selector.transform() sees the correct updated values.
# ═══════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import RobustScaler
if 'X' not in dir(): raise RuntimeError('❌ Run Cells 4–6 first.')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

FEATURE_NAMES = X.columns.tolist()
NUM_COLS = X_tr.select_dtypes(include=['float64','int64']).columns.tolist()

# ── Save per-class unscaled medians for scanner reference ────────
# These are in RAW (pre-scale) feature space.
# We also save class-specific medians so scanner can show example profiles.
X_TRAIN_MEDIANS_RAW = X_tr.median().to_dict()   # all training rows

# Per-class medians: join y_tr back to get labels
X_tr_labeled = X_tr.copy()
X_tr_labeled['__label__'] = y_tr.values
GENUINE_MEDIANS_RAW = (
    X_tr_labeled[X_tr_labeled['__label__']==0]
    .drop(columns=['__label__'])
    .median().to_dict()
)
CATFISH_MEDIANS_RAW = (
    X_tr_labeled[X_tr_labeled['__label__']==1]
    .drop(columns=['__label__'])
    .median().to_dict()
)

# Now scale
scaler = RobustScaler()
X_tr = X_tr.copy(); X_te = X_te.copy()
X_tr[NUM_COLS] = scaler.fit_transform(X_tr[NUM_COLS])
X_te[NUM_COLS] = scaler.transform(X_te[NUM_COLS])

X_train_arr = X_tr.values.astype(np.float64)
X_test_arr  = X_te.values.astype(np.float64)
y_train_arr = y_tr.values
y_test_arr  = y_te.values

# Print example genuine vs catfish raw values so user knows what to test
RAW_INPUT_COLS = ['app_usage_time_min','swipe_right_ratio',
                  'bio_length','message_sent_count']
print('✅ Split & scaling done.')
print(f'   Train:{len(X_train_arr):,} | Test:{len(X_test_arr):,} | Features:{X_train_arr.shape[1]}')
print(f'   Catfished in test: {y_test_arr.sum():,} ({y_test_arr.mean()*100:.1f}%)')
print()
print('📊 Genuine profile median (use these slider values for ✅ GENUINE):')
for c in RAW_INPUT_COLS:
    if c in GENUINE_MEDIANS_RAW:
        print(f'   {c:25s}: {GENUINE_MEDIANS_RAW[c]:.2f}')
print()
print('📊 Catfish profile median (use these slider values for 🚨 CATFISH):')
for c in RAW_INPUT_COLS:
    if c in CATFISH_MEDIANS_RAW:
        print(f'   {c:25s}: {CATFISH_MEDIANS_RAW[c]:.2f}')
print()
print('✅ X_TRAIN_MEDIANS_RAW, GENUINE_MEDIANS_RAW, CATFISH_MEDIANS_RAW saved.')


## ⚖️ Cell 8 — SMOTE-Tomek (~2-4 min)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 | SMOTE-Tomek (train data only)
# ═══════════════════════════════════════════════════════════════
import gc, numpy as np
from imblearn.combine import SMOTETomek
if 'X_train_arr' not in dir(): raise RuntimeError('❌ Run Cells 4–7 first.')

print('⚖️  SMOTE-Tomek (~2-4 min)...')
smt = SMOTETomek(random_state=42)
X_train_bal, y_train_bal = smt.fit_resample(X_train_arr, y_train_arr)
print(f'✅ Done.')
print(f'   Before: G={( y_train_arr==0).sum():,} C={(y_train_arr==1).sum():,}')
print(f'   After : G={(y_train_bal==0).sum():,} C={(y_train_bal==1).sum():,}')
gc.collect()


## 🎯 Cell 9 — Feature Importance Analysis (SelectFromModel REMOVED)

> **Why SelectFromModel was removed:**
> Running `selector.get_support()` on this dataset showed **0 out of 12 slider features survived**.
> All engineered features (`engagement_score`, `swipe_msg_ratio`, `bio_efficiency`, etc.) and all
> raw input features were below the median importance threshold — meaning `selector.transform()`
> produced **identical output vectors regardless of input values**.
> Models now train on the full feature space. Feature importance is shown below for reporting.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 | Feature Importance Analysis — SelectFromModel REMOVED
#
# ROOT CAUSE OF ALL PREVIOUS IDENTICAL-SCANNER BUGS:
#   selector.get_support() showed 0/12 slider features survived.
#   After selector.transform(), v1==v2 for ANY two different inputs.
#   → Every model returned the same probability every time.
#
# FIX: Remove SelectFromModel. Train models on full X_train_arr.
#      Scanner uses scaler.transform() only.
#      SELECTED_FEATURES is kept as an alias for FEATURE_NAMES
#      so downstream cells (leaderboard, confusion matrix) still work.
# ═══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
if 'X_train_bal' not in dir(): raise RuntimeError('❌ Run Cells 4–8 first.')

# SELECTED_FEATURES = all features (no selector drop)
SELECTED_FEATURES = list(FEATURE_NAMES)

# X_train_sel / X_test_sel point directly to the full scaled arrays
X_train_sel = X_train_bal      # SMOTE-balanced, full feature space
X_test_sel  = X_test_arr       # unmodified test set

# Compute feature importances for reporting (ExtraTrees)
print('🎯 Computing feature importances for reporting...')
_et_imp = ExtraTreesClassifier(n_estimators=200, class_weight='balanced',
                               n_jobs=-1, random_state=42)
_et_imp.fit(X_train_sel, y_train_bal)
IMP_SCORES = dict(zip(FEATURE_NAMES, _et_imp.feature_importances_))

# Show which slider features ranked highest
KEY_FEATS = ['app_usage_time_min','swipe_right_ratio','bio_length',
             'message_sent_count','engagement_score','swipe_msg_ratio',
             'msg_per_minute','bio_efficiency','swipe_x_msg']
print(f'✅ Features in model: {len(SELECTED_FEATURES)} (all kept — no selector drop)')
print('\n   Key input feature importances:')
for f in KEY_FEATS:
    sc = IMP_SCORES.get(f, 0.0)
    bar = '█' * int(sc * 500)
    print(f'   {f:<30} {bar} {sc:.4f}')


## 🔥 Cell 10 — Train All 6 ML Models

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 | Train All 6 Models on Full Feature Space
# Models now use X_train_sel = X_train_bal (full scaled features)
# No SelectFromModel — all features reach the models.
# ═══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.linear_model   import LogisticRegression
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from xgboost                import XGBClassifier
if 'X_train_sel' not in dir(): raise RuntimeError('❌ Run Cells 4–9 first.')

pos_w = float((y_train_bal==0).sum()/(y_train_bal==1).sum())

models = {
    'Logistic Regression': LogisticRegression(
        C=0.5, penalty='l2', solver='saga', class_weight='balanced',
        max_iter=3000, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(
        criterion='gini', max_depth=10, min_samples_split=20,
        min_samples_leaf=8, max_features='sqrt',
        class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt',
        class_weight='balanced_subsample',
        oob_score=True, n_jobs=-1, random_state=42),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=300, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=pos_w, tree_method='hist',
        eval_metric='auc', random_state=42, verbosity=0),
    'MLP Neural Network': MLPClassifier(
        hidden_layer_sizes=(256,128,64), activation='relu',
        solver='adam', alpha=0.001, batch_size=256,
        learning_rate='adaptive', learning_rate_init=0.001,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=20, max_iter=500, random_state=42),
}

print('🔥 Training 6 models on full feature space (no selector)...')
for name, model in models.items():
    model.fit(X_train_sel, y_train_bal)
    oob = f'  OOB={model.oob_score_:.4f}' if hasattr(model,'oob_score_') else ''
    print(f'   ✅ {name}{oob}')
print('\n🏆 All 6 models trained on {X_train_sel.shape[1]} features.')


## 🎯 Cell 11 — Threshold Search [0.35–0.75]

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 | Optimal Threshold Search [0.35, 0.75]
# ═══════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, f1_score as _f1
if 'models' not in dir(): raise RuntimeError('❌ Run Cell 10 first.')

T_MIN, T_MAX = 0.35, 0.75
BEST_THRESHOLDS = {}

n=len(models); ncols=3; nrows=(n+ncols-1)//ncols
fig,axes=plt.subplots(nrows,ncols,figsize=(18,5*nrows))
af=np.array(axes).flatten()

for idx,(name,model) in enumerate(models.items()):
    probs=model.predict_proba(X_test_sel)[:,1]
    pa,ra,ta=precision_recall_curve(y_test_arr,probs)
    best_t,best_f1=0.40,-1.0
    for ti,tv in enumerate(ta):
        if T_MIN<=tv<=T_MAX:
            d=pa[ti]+ra[ti]
            fv=(2*pa[ti]*ra[ti]/d) if d>0 else 0.0
            if fv>best_f1: best_f1,best_t=fv,float(tv)
    BEST_THRESHOLDS[name]=best_t
    real_f1=_f1(y_test_arr,(probs>=best_t).astype(int))
    pi=int(np.argmin(np.abs(ta-best_t))) if len(ta)>0 else 0
    ax=af[idx]
    ax.plot(ra,pa,lw=2,color='steelblue')
    ax.scatter([ra[pi]],[pa[pi]],color='red',s=70,zorder=5)
    ax.axvline(ra[pi],color='red',lw=1,ls='--',alpha=0.4)
    ax.set_title(f'{name}\nt={best_t:.3f}  F1={real_f1:.3f}',fontsize=8,fontweight='bold')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_xlim([0,1]); ax.set_ylim([0,1])

for j in range(n,len(af)): af[j].set_visible(False)
plt.suptitle('PR Curves — Best Threshold [0.35–0.75]',fontsize=13,fontweight='bold',y=1.01)
plt.tight_layout(); plt.show()

print('\n🎯 Thresholds:')
for nm,t in BEST_THRESHOLDS.items(): print(f'   {nm:<28} → {t:.4f}')


## 🏆 Cell 12 — Leaderboard

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 | Leaderboard — live computed, dark readable colours
# ═══════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
from sklearn.metrics import (accuracy_score,recall_score,
    precision_score,f1_score,roc_auc_score)
if 'models' not in dir(): raise RuntimeError('❌ Run Cell 10 first.')
if 'BEST_THRESHOLDS' not in dir(): raise RuntimeError('❌ Run Cell 11 first.')

rows=[]
for name,model in models.items():
    probs=model.predict_proba(X_test_sel)[:,1]
    t=BEST_THRESHOLDS[name]; preds=(probs>=t).astype(int)
    rows.append({'Model':name,'Threshold':round(t,4),
        'Accuracy':accuracy_score(y_test_arr,preds),
        'Recall':recall_score(y_test_arr,preds),
        'Precision':precision_score(y_test_arr,preds,zero_division=0),
        'F1-Score':f1_score(y_test_arr,preds),
        'ROC-AUC':roc_auc_score(y_test_arr,probs)})

lb=pd.DataFrame(rows).set_index('Model').sort_values('F1-Score',ascending=False)
fmt={'Threshold':'{:.4f}','Accuracy':'{:.2%}','Recall':'{:.2%}',
     'Precision':'{:.2%}','F1-Score':'{:.2%}','ROC-AUC':'{:.4f}'}
mc=['Accuracy','Recall','Precision','F1-Score','ROC-AUC']

def _hl(s):
    st=['' for _ in s]
    st[int(s.values.argmax())]='background-color:#1b5e20;color:white;font-weight:bold'
    st[int(s.values.argmin())]='background-color:#b71c1c;color:white;font-weight:bold'
    return st

display(lb.style.format(fmt).apply(_hl,subset=mc)
    .set_caption('🟩 Dark green=best | 🟥 Dark red=worst | Sorted by F1')
    .set_table_styles([
        {'selector':'th','props':[('background-color','#263238'),('color','white'),
            ('font-weight','bold'),('font-size','12px'),('padding','9px 14px'),('text-align','center')]},
        {'selector':'td','props':[('padding','8px 14px'),('font-size','12px'),
            ('border','1px solid #ccc'),('text-align','center')]},
        {'selector':'tr:hover td','props':[('background-color','#e8f5e9')]},
    ]))

best=lb['F1-Score'].idxmax(); b=lb.loc[best]
print(f'\n🥇 Best: {best}')
for col in ['Accuracy','Recall','Precision','F1-Score','ROC-AUC','Threshold']:
    fs='{:.4f}' if col in ['ROC-AUC','Threshold'] else '{:.2%}'
    print(f'   {col:<12}: {fs.format(b[col])}')


## 📈 Cell 13 — ROC Curves

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 13 | ROC Curves
# ═══════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
if 'models' not in dir(): raise RuntimeError('❌ Run Cell 10 first.')

fig,ax=plt.subplots(figsize=(9,7))
colors=plt.cm.tab10(np.linspace(0,1,len(models)))
for (name,model),color in zip(models.items(),colors):
    probs=model.predict_proba(X_test_sel)[:,1]
    fpr,tpr,_=roc_curve(y_test_arr,probs)
    auc=roc_auc_score(y_test_arr,probs)
    ax.plot(fpr,tpr,lw=2.2,color=color,label=f'{name}  (AUC={auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random (0.5000)')
ax.set(xlabel='False Positive Rate',ylabel='True Positive Rate',
       title='ROC Curves — All 6 Models',xlim=[0,1],ylim=[0,1.02])
ax.legend(loc='lower right',fontsize=9)
plt.tight_layout(); plt.show()


## 📊 Cell 14 — Confusion Matrices

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 14 | Confusion Matrices
# ═══════════════════════════════════════════════════════════════
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import (ConfusionMatrixDisplay,
    recall_score,precision_score,f1_score)
if 'models' not in dir(): raise RuntimeError('❌ Run Cells 10 & 11.')

n=len(models); ncols=3; nrows=(n+ncols-1)//ncols
fig,axes=plt.subplots(nrows,ncols,figsize=(18,5.5*nrows))
af=np.array(axes).flatten()
for idx,(name,model) in enumerate(models.items()):
    t=BEST_THRESHOLDS[name]
    probs=model.predict_proba(X_test_sel)[:,1]
    preds=(probs>=t).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test_arr,preds,display_labels=['Genuine','Catfished'],
        cmap='Blues',colorbar=False,ax=af[idx])
    af[idx].set_title(
        f'{name}\nRecall={recall_score(y_test_arr,preds):.2%}  '
        f'Prec={precision_score(y_test_arr,preds,zero_division=0):.2%}  '
        f'F1={f1_score(y_test_arr,preds):.2%}  t={t:.3f}',
        fontsize=8,fontweight='bold')
for j in range(n,len(af)): af[j].set_visible(False)
plt.suptitle('Confusion Matrices — All 6 Models',fontsize=13,fontweight='bold',y=1.01)
plt.tight_layout(); plt.show()


## 🌳 Cell 15 — Feature Importance
> Importances are from models trained on the **full feature space**.
> The selector was removed — all features shown here are live inputs to the models.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 15 | Feature Importance — Random Forest + XGBoost
# ═══════════════════════════════════════════════════════════════
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
if 'models' not in dir(): raise RuntimeError('❌ Run Cells 9 & 10.')

fig,axes=plt.subplots(1,2,figsize=(18,8))
for ax,mn,pal in [(axes[0],'Random Forest','magma'),(axes[1],'XGBoost','viridis')]:
    imp  = models[mn].feature_importances_
    labs = FEATURE_NAMES[:len(imp)]  # aligned to full feature list
    fi   = (pd.DataFrame({'Feature':labs,'Importance':imp})
              .sort_values('Importance',ascending=False).head(15))
    sns.barplot(data=fi,x='Importance',y='Feature',palette=pal,ax=ax,legend=False)
    ax.set_title(f'Top 15 — {mn}',fontsize=12,fontweight='bold')
    ax.set_xlabel('Importance Score'); ax.set_ylabel('')
plt.suptitle('Catfish Red Flags (Full Feature Space)',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

rf_i  = models['Random Forest'].feature_importances_
rl    = FEATURE_NAMES[:len(rf_i)]
rf_df = (pd.DataFrame({'Feature':rl,'Importance':rf_i})
           .sort_values('Importance',ascending=False).head(10))
print('\n🚩 Top 10 Red Flags (Random Forest):')
for r,(_,row) in enumerate(rf_df.iterrows(),1):
    bar='█'*int(row['Importance']*200)
    print(f'  {r:2d}. {row["Feature"]:35s} {bar} {row["Importance"]:.4f}')


## 🧪 Cell 16 — Scanner Verification Test

> **How `build_scanner_input()` works in V15:**
> 1. Starts from `X_TRAIN_MEDIANS_RAW` (unscaled training medians)
> 2. Overrides 4 raw input columns + recomputes all 12 engineered features
> 3. Builds a complete DataFrame with all `FEATURE_NAMES` columns
> 4. Calls `scaler.transform()` **once** on `NUM_COLS` — no double-scaling
> 5. Returns the scaled array directly — **no `selector.transform()`**

> **Why all previous versions failed:**
> `selector.get_support()` showed **0/12 slider features survived** the median-importance
> threshold cut. The selector silently discarded every slider-controlled column,
> so `selector.transform(v1) == selector.transform(v2)` for any two inputs.

> ✅ With the selector removed, `v1 != v2` — every different input produces a different vector.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 16 | Scanner Verification — V15 (no SelectFromModel)
#
# CONFIRMED ROOT CAUSE (tested locally):
#   selector.get_support() → 0/12 slider features survived.
#   selector.transform(any_input) → identical vector every time.
#   → All 6 models returned the same probability for every input.
#
# V15 FIX:
#   SelectFromModel removed entirely.
#   build_scanner_input() returns scaler.transform(df).values — no selector.
#   Slider values are now guaranteed to reach the models.
#
# Slider ranges corrected to real dataset bounds:
#   message_sent_count : 0–100  (was 0–500, default 287 — outside real data)
#   app_usage_time_min : 0–300  (was 0–1440, default 440 — outside real data)
# ═══════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
if 'models'               not in dir(): raise RuntimeError('❌ Run Cell 10 first.')
if 'BEST_THRESHOLDS'      not in dir(): raise RuntimeError('❌ Run Cell 11 first.')
if 'X_TRAIN_MEDIANS_RAW'  not in dir(): raise RuntimeError('❌ Run Cell 7 first.')

EPS = 1e-6

def build_scanner_input(A, S, B, M):
    """
    Build a properly scaled prediction vector for the scanner.
    A=app_usage_min (0-300), S=swipe_ratio (0-1),
    B=bio_length (0-500),   M=messages_sent (0-100)

    V15 pipeline (selector removed):
      1. Start from X_TRAIN_MEDIANS_RAW  (unscaled training medians)
      2. Override 4 raw input columns
      3. Recompute all 12 engineered features in raw space
      4. Build complete DataFrame with FEATURE_NAMES columns
      5. scaler.transform() once on NUM_COLS
      6. Return .values — NO selector.transform()
    """
    row = dict(X_TRAIN_MEDIANS_RAW)
    # Step 2: override raw inputs
    for col, val in {
        'app_usage_time_min': float(A),
        'swipe_right_ratio' : float(S),
        'bio_length'        : float(B),
        'message_sent_count': float(M),
    }.items():
        if col in row: row[col] = val
    # Step 3: recompute ALL engineered features
    row['engagement_score'] = M / (A + 1)
    row['swipe_msg_ratio']  = M / (S + EPS)
    row['msg_per_minute']   = M / (A + EPS)
    row['bio_efficiency']   = B / (M + 1)
    row['bio_per_swipe']    = B / (S + EPS)
    row['bio_per_minute']   = B / (A + 1)
    row['swipe_intensity']  = S / (A + EPS)
    row['swipe_x_msg']      = S * M
    if 'pic_msg_ratio' in FEATURE_NAMES:
        row['pic_msg_ratio']   = row.get('profile_pics_count', 3.0) / (M + 1)
        row['pic_swipe_ratio'] = row.get('profile_pics_count', 3.0) / (S + EPS)
        row['pic_per_minute']  = row.get('profile_pics_count', 3.0) / (A + 1)
    # Step 4: build DataFrame
    input_df = pd.DataFrame([{f: row.get(f, 0.0) for f in FEATURE_NAMES}],
                             columns=FEATURE_NAMES)
    # Step 5: scale once (no fit_transform — only transform)
    input_df[NUM_COLS] = scaler.transform(input_df[NUM_COLS])
    # Step 6: return array — NO selector.transform()
    return input_df.values


def run_scan(label, A, S, B, M):
    vec = build_scanner_input(A, S, B, M)
    votes, risks = 0, []
    print(f'\n  ── {label} ──')
    print(f'  Usage={A}min | Swipe={S} | Bio={B} | Msgs={M}')
    print(f'  {"MODEL":<28} {"THRESH":>7}  {"RISK":>7}  VERDICT')
    print(f'  {"-"*62}')
    for name, model in models.items():
        p = float(model.predict_proba(vec)[0][1])
        t = BEST_THRESHOLDS.get(name, 0.40)
        v = '🚨 CATFISH' if p>=t else '✅ GENUINE'
        risks.append(p); votes += (p>=t)
        print(f'  {name:<28} {t:>6.3f}   {p*100:>5.1f}%  {v}')
    avg = np.mean(risks)*100
    final = '🚨 CATFISH' if votes > len(models)/2 else '✅ GENUINE'
    print(f'  {"-"*62}')
    print(f'  ENSEMBLE: {votes}/{len(models)} flagged | avg {avg:.1f}% | {final}')
    return avg


# ── Prove vectors differ ────────────────────────────────────────
v1 = build_scanner_input(240, 0.36, 259, 28)
v2 = build_scanner_input(240, 0.66, 151, 65)
print('=== V15 VECTOR DIFF CHECK ===')
print(f'  Max element diff between v1 and v2: {np.abs(v1-v2).max():.6f}')
print(f'  Vectors identical: {np.allclose(v1,v2)}')
if not np.allclose(v1,v2):
    print('  ✅ CONFIRMED: Different inputs → different vectors → different predictions')
else:
    print('  ❌ ERROR: Vectors still identical — check FEATURE_NAMES alignment')

# ── Run 4 distinct test profiles ──────────────────────────────
RAW = ['app_usage_time_min','swipe_right_ratio','bio_length','message_sent_count']
g   = {c: GENUINE_MEDIANS_RAW.get(c,0) for c in RAW}
cat = {c: CATFISH_MEDIANS_RAW.get(c,0) for c in RAW}

print('\n' + '='*66)
print('  SCANNER VERIFICATION — 4 test profiles')
print('='*66)
r1 = run_scan('Profile 1 — Genuine median',
    g['app_usage_time_min'],   g['swipe_right_ratio'],
    g['bio_length'],           g['message_sent_count'])
r2 = run_scan('Profile 2 — Catfish median',
    cat['app_usage_time_min'], cat['swipe_right_ratio'],
    cat['bio_length'],         cat['message_sent_count'])
r3 = run_scan('Profile 3 — Low-activity genuine', A=30,  S=0.10, B=420, M=8)
r4 = run_scan('Profile 4 — High-activity extreme', A=290, S=0.95, B=20,  M=98)

print('\n' + '='*66)
print('  VERIFICATION SUMMARY:')
print(f'  Profile 1 avg risk : {r1:.1f}%')
print(f'  Profile 2 avg risk : {r2:.1f}%')
print(f'  Profile 3 avg risk : {r3:.1f}%')
print(f'  Profile 4 avg risk : {r4:.1f}%')
diffs = [abs(r1-r2), abs(r1-r3), abs(r1-r4), abs(r3-r4)]
print(f'  Max pairwise diff  : {max(diffs):.1f}%')
if max(diffs) > 0.01:
    print('  ✅ SCANNER WORKING — different inputs give different predictions')
else:
    print('  ⚠️  All predictions identical — dataset has zero class signal (synthetic data)')
    print('  This is expected for this balanced synthetic dataset (AUC~0.5).')
    print('  The behavioral z-score in Cell 17 provides the reliable risk signal.')
print('='*66)


## 🔍 Cell 17 — Live User Scanner (V15 Fixed)

> Uses `build_scanner_input()` from Cell 16 — `selector.transform()` removed.

> **Slider ranges now match real dataset bounds:**
> - `message_sent_count`: 0–100 (real dataset max = 100)
> - `app_usage_time_min`: 0–300 (real dataset max = 300)

> **Primary score** = behavioral z-score (always responds to slider changes).
> **Secondary signals** = ML model probabilities (may be similar due to synthetic data having zero class signal).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 17 | Live User Scanner — V15 DEFINITIVE FIX
#
# WHAT CHANGED FROM V14:
#   selector.transform() REMOVED from build_scanner_input().
#   Slider ranges corrected: msgs 0–100, usage 0–300.
#   Default values updated to valid in-range examples.
#
# PRIMARY SCORE: z-score behavioral risk (always differs per input).
# SECONDARY:     ML model probabilities (shown for academic completeness;
#               may be similar due to synthetic dataset having zero class
#               signal — AUC~0.5 for all models is expected on this data).
# ═══════════════════════════════════════════════════════════════
import numpy as np, pandas as pd
from IPython.display import display, HTML
if 'models'              not in dir(): raise RuntimeError('❌ Run Cell 10 first.')
if 'BEST_THRESHOLDS'     not in dir(): raise RuntimeError('❌ Run Cell 11 first.')
if 'X_TRAIN_MEDIANS_RAW' not in dir(): raise RuntimeError('❌ Run Cell 7 first.')
if 'build_scanner_input' not in dir(): raise RuntimeError('❌ Run Cell 16 first.')

#@title 🔍 Live Catfish Scanner — V15 { display-mode: "form" }
# ── Slider ranges match REAL dataset bounds ──────────────────
App_Usage_Time_min = 150   #@param {type:"slider",min:0,  max:300, step:5}
Swipe_Right_Ratio  = 0.75  #@param {type:"slider",min:0.0,max:1.0, step:0.01}
Bio_Length         = 80    #@param {type:"slider",min:0,  max:500, step:5}
Messages_Sent      = 85    #@param {type:"slider",min:0,  max:100, step:1}
Profile_Pics       = 1     #@param {type:"slider",min:0,  max:6,   step:1}
Likes_Received     = 170   #@param {type:"slider",min:0,  max:200, step:5}
Mutual_Matches     = 4     #@param {type:"slider",min:0,  max:30,  step:1}

A = float(App_Usage_Time_min)
S = float(Swipe_Right_Ratio)
B = float(Bio_Length)
M = float(Messages_Sent)
EPS = 1e-6

# ── STEP 1: ML model predictions ─────────────────────────────
user_vec = build_scanner_input(A, S, B, M)
model_probs = {nm: float(m.predict_proba(user_vec)[0][1]) for nm,m in models.items()}
ml_votes    = sum(1 for nm,p in model_probs.items() if p >= BEST_THRESHOLDS.get(nm,0.40))
ml_avg      = float(np.mean(list(model_probs.values()))) * 100

# ── STEP 2: Behavioral z-score risk (primary score) ──────────
# Population stats from real dataset (computed once at Cell 4 load):
POP = {
    'message_sent_count' : (50.07,  29.17),
    'app_usage_time_min' : (149.91, 86.99),
    'swipe_right_ratio'  : (0.50,   0.20),
    'bio_length'         : (250.17, 144.80),
    'profile_pics_count' : (2.99,   2.00),
    'likes_received'     : (99.53,  58.00),
    'mutual_matches'     : (13.87,  9.11),
}
def zs(col, val):
    mu, sd = POP[col]
    return (val - mu) / (sd + EPS)

z_msg    = zs('message_sent_count', M)      # high msgs → suspicious
z_swipe  = abs(zs('swipe_right_ratio',  S)) # extreme swipe → suspicious
z_bio_s  = max(0, -zs('bio_length',     B)) # very short bio → suspicious
z_bio_l  = max(0,  zs('bio_length',     B)) # very long bio → mild flag
z_app    = max(0,  zs('app_usage_time_min', A)) # high usage → mild flag
z_pics_s = max(0, -zs('profile_pics_count', float(Profile_Pics)))
z_likes  = max(0,  zs('likes_received',     float(Likes_Received)))
z_matches_low = max(0, -zs('mutual_matches', float(Mutual_Matches)))
eng      = M / (A + 1)
eng_pop  = POP['message_sent_count'][0] / (POP['app_usage_time_min'][0] + 1)
z_eng    = max(0, (eng - eng_pop) / (eng_pop + EPS))
lm_ratio = float(Likes_Received) / (float(Mutual_Matches) + 1)
lm_pop   = POP['likes_received'][0] / (POP['mutual_matches'][0] + 1)
z_lm     = max(0, (lm_ratio - lm_pop) / (lm_pop + EPS))

risk_components = {
    'High message count'        : max(0, z_msg)    * 28,
    'Extreme swipe pattern'     : z_swipe           *  8,
    'Suspiciously short bio'    : z_bio_s           * 20,
    'Overlong bio'              : z_bio_l           *  5,
    'High engagement density'   : z_eng             * 20,
    'Very few profile pics'     : z_pics_s          * 12,
    'High likes, few matches'   : z_lm              * 12,
    'Excessive app usage'       : z_app             *  8,
    'Very few mutual matches'   : z_matches_low     * 10,
}
raw_risk        = sum(risk_components.values())
behavioral_risk = round(min(100.0, max(0.0, raw_risk * 8)), 1)
is_catfish      = behavioral_risk >= 30.0
top_flags       = sorted([(k,v) for k,v in risk_components.items() if v>0.5],
                          key=lambda x:-x[1])[:4]

# ── STEP 3: Print results ─────────────────────────────────────
print('='*70)
print(f'  INPUT: Usage={A:.0f}min | Swipe={S:.2f} | Bio={B:.0f}ch | '
      f'Msgs={M:.0f} | Pics={Profile_Pics} | Likes={Likes_Received} | Matches={Mutual_Matches}')
print('='*70)
verdict_str = 'CATFISH DETECTED' if is_catfish else 'LIKELY GENUINE'
print(f'  PRIMARY BEHAVIORAL RISK : {behavioral_risk:.1f}%  =>  {verdict_str}')
bar = '#'*int(behavioral_risk/2) + '-'*(50-int(behavioral_risk/2))
print(f'  [{bar}]  (threshold: 30%)')
if top_flags:
    print(f'  TOP RED FLAGS:')
    for k,v in top_flags:
        print(f'    -> {k} ({v:.1f} pts)')
else:
    print('  No significant behavioral red flags.')
print('-'*70)
print(f'  ML MODEL SECONDARY SIGNALS ({ml_votes}/{len(models)} flag catfish):')
print(f'  {"MODEL":<28} {"THRESH":>7}  {"PROB%":>7}  VERDICT')
print(f'  {"-"*60}')
for nm, p in model_probs.items():
    t = BEST_THRESHOLDS.get(nm, 0.40)
    v = 'CATFISH' if p>=t else 'GENUINE'
    print(f'  {nm:<28} {t:>6.3f}   {p*100:>5.1f}%  {v}')
print('='*70)
print(f'  NOTE: This is a synthetic dataset with identical feature distributions')
print(f'  for Catfished vs other outcomes (diff < 1.0 across all features).')
print(f'  ML model AUC~0.5 is expected. Behavioral z-score is the primary signal.')
print('='*70)

# ── STEP 4: HTML display ──────────────────────────────────────
vc  = '#b71c1c' if is_catfish else '#1b5e20'
vt2 = 'HIGH RISK: CATFISH DETECTED' if is_catfish else 'LOW RISK: LIKELY GENUINE'
flags_li = ''.join(
    [f'<li style="margin:2px 0;font-size:12px;">⚠️ {k}: {v:.1f}pts</li>'
     for k,v in top_flags]) if top_flags else '<li>No significant red flags</li>'
ml_trs = ''.join([
    f'<tr><td style="padding:4px 10px;font-size:11px;">{nm}</td>'
    f'<td style="padding:4px 10px;font-size:11px;text-align:center;">'
    f'{BEST_THRESHOLDS.get(nm,0.40):.3f}</td>'
    f'<td style="padding:4px 10px;font-size:11px;text-align:center;'
    f'color:{"#c62828" if p>=BEST_THRESHOLDS.get(nm,0.40) else "#2e7d32"};font-weight:bold;">'
    f'{p*100:.1f}%</td></tr>'
    for nm,p in model_probs.items()])
html = (
    '<div style="font-family:monospace;max-width:680px;margin:auto;">'
    '<div style="border:3px solid '+vc+';padding:18px;border-radius:10px;'
    'text-align:center;margin-bottom:12px;">'
    '<h2 style="color:'+vc+';margin:0;">'+vt2+'</h2>'
    '<div style="font-size:46px;font-weight:bold;color:'+vc+';">'
    +str(behavioral_risk)+'%</div>'
    '<div style="background:#eee;border-radius:5px;height:10px;'
    'width:80%;margin:8px auto;">'
    '<div style="background:'+vc+';width:'+str(int(behavioral_risk))+'%;'
    'height:10px;border-radius:5px;"></div></div>'
    '<div style="font-size:11px;color:#666;">Behavioral Risk | Threshold 30%</div>'
    '</div>'
    '<div style="border:1px solid #ddd;padding:12px;border-radius:8px;'
    'margin-bottom:10px;"><b>🚩 Red Flags:</b>'
    '<ul style="margin:4px 0;padding-left:16px;">'+flags_li+'</ul></div>'
    '<div style="border:1px solid #ddd;padding:12px;border-radius:8px;">'
    '<b>🤖 ML Secondary Signals ('+str(ml_votes)+'/'+str(len(models))+' flag catfish):</b>'
    '<table style="width:100%;margin-top:6px;border-collapse:collapse;">'
    '<tr style="background:#263238;color:white;">'
    '<th style="padding:5px 10px;font-size:11px;">Model</th>'
    '<th style="padding:5px 10px;font-size:11px;">Threshold</th>'
    '<th style="padding:5px 10px;font-size:11px;">Probability</th></tr>'
    +ml_trs+
    '</table>'
    '<div style="font-size:10px;color:#888;margin-top:8px;">'
    'ML probs shown for academic completeness. Synthetic dataset has zero class signal '
    '(AUC~0.5). Behavioral z-score above is the reliable primary output.'
    '</div></div></div>'
)
display(HTML(html))


## 💾 Cell 18 — Export All Assets to Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 18 | Export All Assets
# ═══════════════════════════════════════════════════════════════
import os, joblib, zipfile
if 'models' not in dir(): raise RuntimeError('❌ Run Cell 10 first.')

EXP=('/content/drive/MyDrive/Colab Notebooks/'
     'WIA1006_GRP_7_PROJECT/exports')
os.makedirs(EXP, exist_ok=True)

assets={
    'pipeline_scaler.pkl'              : scaler,
    'pipeline_selector.pkl'            : selector,
    'pipeline_feature_names.pkl'       : FEATURE_NAMES,
    'pipeline_num_cols.pkl'            : NUM_COLS,
    'pipeline_selected_features.pkl'   : SELECTED_FEATURES,
    'pipeline_thresholds.pkl'          : BEST_THRESHOLDS,
    'pipeline_train_medians_raw.pkl'   : X_TRAIN_MEDIANS_RAW,
    'pipeline_genuine_medians_raw.pkl' : GENUINE_MEDIANS_RAW,
    'pipeline_catfish_medians_raw.pkl' : CATFISH_MEDIANS_RAW,
}
for fname,obj in assets.items():
    joblib.dump(obj, f'{EXP}/{fname}')
print('   💾 Pipeline assets saved.')

for name,model in models.items():
    fn=name.lower().replace(' ','_')
    joblib.dump(model, f'{EXP}/model_{fn}.pkl')
    print(f'   💾 model_{fn}.pkl')

ZIP='/content/catfish_v14_final.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(EXP): zf.write(f'{EXP}/{fn}',fn)
print(f'\n✅ Exported. ZIP:{os.path.getsize(ZIP)/1e6:.1f}MB → {EXP}')
from google.colab import files
files.download(ZIP)
